In [1]:
from pathlib import Path
import os
import sys
import gc
import json
from typing import Any

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
    
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CATEGORIES

CATEGORY = 'fantasy_paranormal'
cfg = CATEGORIES[CATEGORY]
INTERIM_DIR = cfg.processed_dir

BOOKS_IN_PATH = INTERIM_DIR / 'books_clean.parquet'
INTER_IN_PATH = INTERIM_DIR / 'interactions_clean.parquet'

BOOKS_OUT_PATH = INTERIM_DIR / 'books_reduced.parquet'
INTER_OUT_PATH = INTERIM_DIR / 'interactions_reduced.parquet'

In [2]:
books = pd.read_parquet(BOOKS_IN_PATH)
interactions = pd.read_parquet(INTER_IN_PATH)

print(f'Books: {len(books):,}')
print(f'Interactions: {len(interactions):,}')

Books: 258,585
Interactions: 55,397,550


## Feature engineering for nested features

In [3]:
def json_dumps_or_na(value: Any) -> Any:
    if value is None:
        return pd.NA
    if not isinstance(value, (list, dict)) and pd.isna(value):
        return pd.NA
    return json.dumps(value, ensure_ascii=False)

def features_from_nested(series: pd.Series, extractor) -> pd.DataFrame:
    return pd.DataFrame(series.map(extractor).tolist(), index=series.index)

def has_theme(names: list[str], keywords: list[str]) -> bool:
    return any(any(keyword in name for keyword in keywords) for name in names)

def extract_author_features(authors: Any) -> dict[str, Any]:
    if not isinstance(authors, list):
        return {
            'primary_author_id_role_filtered': pd.NA,
            'author_fallback_id': pd.NA,
            'author_count': 0,
            'non_primary_role_count': 0,
            'primary_author_role': pd.NA,
        }
    valid = [item for item in authors if isinstance(item, dict) and item.get('author_id')]
    role_filtered = [item for item in valid if str(item.get('role', '')) == '']
    fallback = valid[0] if valid else None
    primary = role_filtered[0] if role_filtered else None
    return {
        'primary_author_id_role_filtered': str(primary.get('author_id')) if primary else pd.NA,
        'author_fallback_id': str(fallback.get('author_id')) if fallback else pd.NA,
        'author_count': len(valid),
        'non_primary_role_count': sum(1 for item in valid if str(item.get('role', '')) != ''),
        'primary_author_role': str(primary.get('role', '')) if primary else pd.NA,
    }


## Cleaning and flatenning

In [5]:
print('--- Book cleaning ---')
books['book_id'] = books['book_id'].astype(str)
if 'title' in books.columns:
    books = books[books['title'].notna()].copy()

if 'publication_year' in books.columns:
    pub_year = pd.to_numeric(books['publication_year'], errors='coerce')
    books['publication_year'] = pub_year.where(pub_year.between(1450, 2026))
    
if 'language_code' in books.columns:
    eng_codes = {'eng', 'en-US', 'en-GB', 'en-CA'}
    books = books[books['language_code'].isin(eng_codes) | books['language_code'].isna()]

# Nested features
if 'authors' in books.columns:
    print('Processing authors...')
    author_features = features_from_nested(books['authors'], extract_author_features)
    books = pd.concat([books, author_features], axis=1)
    books = books.drop(columns=['authors'])

print(f'Valid books: {len(books):,}')

print('\n--- Interaction cleaning ---')
interactions['book_id'] = interactions['book_id'].astype(str)
interactions['user_id'] = interactions['user_id'].astype(str)

if 'rating' in interactions.columns:
    rating = pd.to_numeric(interactions['rating'], errors='coerce')
    interactions['rating'] = rating.where(rating.between(1, 5))
    interactions = interactions[interactions['rating'].notna()].copy()

print(f'Valid interactions: {len(interactions):,}')

--- Book cleaning ---
Valid books: 147,405

--- Interaction cleaning ---
Valid interactions: 26,193,771


## K-Core Filtering

In [6]:
def apply_k_core(inter_df, books_df, k_book, k_user):
    print(f"\nTesting K_book={k_book}, K_user={k_user}...")
    valid_books = set(books_df['book_id'])
    valid_users = set(inter_df['user_id'])
    
    filtered_inter = inter_df[inter_df['book_id'].isin(valid_books)].copy()
    
    for iteration in range(20):
        prev_books, prev_users = len(valid_books), len(valid_users)
        
        mask = filtered_inter['book_id'].isin(valid_books) & filtered_inter['user_id'].isin(valid_users)
        filtered_inter = filtered_inter[mask]
        
        book_counts = filtered_inter['book_id'].value_counts()
        user_counts = filtered_inter['user_id'].value_counts()
        
        valid_books = set(book_counts[book_counts >= k_book].index)
        valid_users = set(user_counts[user_counts >= k_user].index)
        
        if len(valid_books) == prev_books and len(valid_users) == prev_users:
            print(f'  Converged on {iteration+1} iteration. Books: {len(valid_books):,}, Users: {len(valid_users):,}, Interactions: {len(filtered_inter):,}')
            break
            
    return valid_books, valid_users, filtered_inter

test_cases = [
    (20, 5),
    (30, 5),
    (30, 10),
    (50, 5),
    (100, 5)
]
for kb, ku in test_cases:
    apply_k_core(interactions, books, k_book=kb, k_user=ku)


Testing K_book=20, K_user=5...
  Converged on 6 iteration. Books: 55,518, Users: 462,059, Interactions: 22,340,395

Testing K_book=30, K_user=5...
  Converged on 7 iteration. Books: 43,450, Users: 460,939, Interactions: 22,046,312

Testing K_book=30, K_user=10...
  Converged on 6 iteration. Books: 42,954, Users: 359,352, Interactions: 21,339,398

Testing K_book=50, K_user=5...
  Converged on 6 iteration. Books: 31,281, Users: 459,287, Interactions: 21,575,033

Testing K_book=100, K_user=5...
  Converged on 6 iteration. Books: 19,506, Users: 456,349, Interactions: 20,736,277


In [7]:
CHOSEN_K_BOOK = 50 
CHOSEN_K_USER = 5

final_valid_books, final_valid_users, interactions_reduced = apply_k_core(interactions, books, CHOSEN_K_BOOK, CHOSEN_K_USER)

books_reduced = books[books['book_id'].isin(final_valid_books)].copy()

print(f"\n--- RESULTS ---")
print(f"Books: {len(books_reduced):,}")
print(f"Interactions: {len(interactions_reduced):,}")
print(f"Users: {len(final_valid_users):,}")


Testing K_book=50, K_user=5...
  Converged on 6 iteration. Books: 31,281, Users: 459,287, Interactions: 21,575,033

--- RESULTS ---
Books: 31,281
Interactions: 21,575,033
Users: 459,287


## Saving

In [8]:
books_reduced.to_parquet(BOOKS_OUT_PATH, index=False, engine='pyarrow', compression='snappy')
interactions_reduced.to_parquet(INTER_OUT_PATH, index=False, engine='pyarrow', compression='snappy')

print(f'Sizes: {BOOKS_OUT_PATH.stat().st_size / 1e6:.1f} MB, {INTER_OUT_PATH.stat().st_size / 1e6:.1f} MB')

Sizes: 23.4 MB, 512.5 MB


In [9]:
books = pd.read_parquet(BOOKS_OUT_PATH)
print(books.columns)

Index(['text_reviews_count', 'series', 'language_code', 'popular_shelves',
       'average_rating', 'num_pages', 'publication_year', 'book_id',
       'ratings_count', 'work_id', 'title', 'primary_author_id_role_filtered',
       'author_fallback_id', 'author_count', 'non_primary_role_count',
       'primary_author_role'],
      dtype='str')
